# Cyclone intensity estimation - Colab training notebook
**Runtime > Change runtime type > GPU** before starting.
Every cell is a command from the README. Nothing here produces results until you run it on real data.

## 1. Get the code
Upload `cyclone-intensity-estimation.zip` to Colab (left panel > Files), then run:

In [ ]:
!unzip -q -o cyclone-intensity-estimation.zip
%cd cyclone-intensity-estimation
!pip install -q -r requirements-train.txt

## 2. Software check (optional, ~2 min): synthetic smoke test
Uses ARTIFICIAL images only to verify the code runs. Output checkpoints are named `demo_*` and must never be reported.

In [ ]:
!python -m src.make_smoke_data
!python -m src.data_loader --config config/config_smoke.yaml
!python -m src.train --config config/config_smoke.yaml --model cnn --demo-run
!python -m src.evaluate --checkpoint artifacts/demo_best_cnn.pt --config config/config_smoke.yaml --mc-samples 5

## 3. Real data: download TCIR
Obtain `TCIR.h5` from the dataset page (Chen, Chen & Lin) and upload it to Google Drive. Then mount Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
H5 = '/content/drive/MyDrive/TCIR.h5'   # <- change to your file location

Inspect the file first. Check the printed column names, the wind range (knots vs m/s) and the IR channel statistics.

In [ ]:
!python -m src.prepare_tcir --h5 "$H5" --inspect

Convert (use `--stride 3` or `--max-frames 15000` to keep Colab time reasonable). Add `--storm-col/--time-col/--wind-col` if auto-detection failed, and `--wind-scale 1.94384` if the wind column is in m/s.

In [ ]:
!python -m src.prepare_tcir --h5 "$H5" --out data/raw --stride 3
!python -m src.data_loader --config config/config.yaml

## 4. Train and evaluate (real data)

In [ ]:
!python -m src.train --config config/config.yaml --model cnn
!python -m src.train --config config/config.yaml --model resnet18

In [ ]:
!python -m src.evaluate --checkpoint artifacts/best_cnn.pt --mc-samples 20
!python -m src.evaluate --checkpoint artifacts/best_resnet18.pt --mc-samples 20

## 5. Download the results
Copy `artifacts/` (checkpoints, metrics, figures) back to your computer / Drive.

In [ ]:
!zip -r artifacts.zip artifacts reports
from google.colab import files
files.download('artifacts.zip')